## **Practice: LIME**

In the theory block we went through the construction of LIME at the top level. If we go into the details, then under the hood LIME also does (or can do) interesting transformations that are able to strengthen and specialise the explanation. They are reflected in the hyperparameters. But instead of using the LIME wrapper, in this practical session we will figure everything out and build two surrogate models of our own for scratch And we will recreate the approach from [this](https://arxiv.org/pdf/1910.13016) paper!  

Let us begin!

[![temp-Image1mlt-QZ.avif](https://i.postimg.cc/LXny3rrP/temp-Image1mlt-QZ.avif)](https://postimg.cc/hzRbS344)

In [ ]:
!pip install lime fat-forensics[all] -q #installing the required libraries

In [ ]:
import fatf
import lime
import pandas as pd
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

fatf.setup_random_seed(42)

Let us prepare the data. For simplicity, we will work with a dataset that is very well known in statistics — Fisher's Irises. It consists of measurements taken from 150 iris specimens, 50 specimens of each of three species — Iris setosa, Iris virginica and Iris versicolor.

Four characteristics are given for them (in centimetres):

- The length of the outer perianth lobe (sepal length);
- The width of the outer perianth lobe (sepal width);
- The length of the inner perianth lobe (petal length);
- The width of the inner perianth lobe (petal width).

On the basis of this dataset we have to build a classification rule that determines the species of the plant from the measurements. This is a multiclass classification task, since there are three classes — three iris species.

In [ ]:
#Loading

iris_data_dict = load_iris()
iris_data = iris_data_dict['data']
iris_target = iris_data_dict['target']
iris_feature_names = iris_data_dict['feature_names']
iris_target_names = iris_data_dict['target_names']

X_train, X_test, y_train, y_test = train_test_split(iris_data, iris_target, random_state=42)

We will use a random forest as the "black box" model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import sklearn.metrics

blackbox_model = RandomForestClassifier(random_state=42, max_depth=2)
blackbox_model.fit(X_train, y_train)

predictions = blackbox_model.predict(X_test)
acc = sklearn.metrics.accuracy_score(y_test, predictions)

print(f'Model accuracy: {acc}')

In [ ]:
data_point = X_train[37] #Let us pick an arbitrary data point

data_point_probabilities = blackbox_model.predict_proba(data_point.reshape(1, -1))[0]
data_point_probabilities

In [ ]:
data_point

In [ ]:
data_point_prediction = data_point_probabilities.argmax(axis=0) #We look at the class of the data point

data_point_class = iris_target_names[data_point_prediction]
data_point_class

Great! We are dealing with Iris versicolor. To literally take a look at it, let us visualise the dataset and the chosen data point.

In [ ]:
X_train[y_train==1][:, 2]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


_ = plt.figure()
_ = plt.scatter(
     X_train[y_train==0][:, 2],
     X_train[y_train==0][:, 3],
     label=iris_target_names[0])
_ = plt.scatter(
     X_train[y_train==1][:, 2],
     X_train[y_train==1][:, 3],
     label=iris_target_names[1])
_ = plt.scatter(
     X_train[y_train==2][:, 2],
     X_train[y_train==2][:, 3],
     label=iris_target_names[2])
_ = plt.scatter(
     data_point[2],
     data_point[3],
     label='Explained Data Point',
    s=100, c='k')

_ = plt.xlabel(iris_feature_names[2])
_ = plt.ylabel(iris_feature_names[3])
_ = plt.legend()

_ = plt.title('Observed points and the explained one')

## Build LIME model

Let us recall a few facts from the theory lesson. When training a surrogate model $g(z)$ for a model $f(x)$ we solve a problem of the form:

$$\xi(x) = argmin_{g \in G} L(f, g, \pi_x) + \Omega(g).$$

The loss function $L$ contains a non-standard parameter $\pi_x$. It is responsible for adjusting the contribution of the objects from the neighbourhood, assigning weights to them, and inside the function $L$ it is a certain correction in the computation of the squared error.

$$L(f, g, \pi_x) = \pi_x(f(z) - g(x'))^2,$$

**Quiz 1: How is the weight of each object $\pi_x$ in the loss function minimised when training the surrogate model computed?**

In the original implementation (that is, in the code for the paper [“Why Should I Trust You?” Explaining the Predictions of Any Classifier](https://arxiv.org/pdf/1602.04938)) the local surrogate model can be trained either with *discretisation* of the continuous features or without it, and for some data modalities (text, images) binarisation is used. These notions are not yet familiar to us, so let us go through them.

In [ ]:
import fatf.utils.data.discretisation as fatf_discretisation #importing the helper functions
import fatf.utils.data.augmentation as fatf_augmentation

The first step of the LIME algorithm is the generation of data similar to the original data.

In [ ]:
augmenter = fatf_augmentation.Mixup(X_train, ground_truth=y_train)

sampled_data = augmenter.sample(data_point, samples_number=100) #We generated points on the basis of the original ones
sampled_data_probabilities = blackbox_model.predict_proba(sampled_data) #We predicted the probabilities for the generated points

Let us look at the new points.

In [ ]:
sampled_data_predictions = sampled_data_probabilities.argmax(axis=1)
sampled_data_0_indices = np.where(sampled_data_predictions == 0)[0]
sampled_data_1_indices = np.where(sampled_data_predictions == 1)[0]
sampled_data_2_indices = np.where(sampled_data_predictions == 2)[0]

_ = plt.figure()
_ = plt.scatter(
     X_train[y_train==0][:, 2],
     X_train[y_train==0][:, 3],
     label=iris_target_names[0])
_ = plt.scatter(
     X_train[y_train==1][:, 2],
     X_train[y_train==1][:, 3],
     label=iris_target_names[1])
_ = plt.scatter(
     X_train[y_train==2][:, 2],
     X_train[y_train==2][:, 3],
     label=iris_target_names[2])
_ = plt.scatter(
     data_point[2],
     data_point[3],
     label='Explained Data Point',
    s=100, c='k')


_ = plt.scatter(
     sampled_data[sampled_data_0_indices, 2],
     sampled_data[sampled_data_0_indices, 3],
     label='Augmented Data: {}'.format(iris_target_names[0]))
_ = plt.scatter(
     sampled_data[sampled_data_1_indices, 2],
     sampled_data[sampled_data_1_indices, 3],
     label='Augmented Data: {}'.format(iris_target_names[1]))
_ = plt.scatter(
     sampled_data[sampled_data_2_indices, 2],
     sampled_data[sampled_data_2_indices, 3],
     label='Augmented Data: {}'.format(iris_target_names[2]))

_ = plt.xlabel(iris_feature_names[2])
_ = plt.ylabel(iris_feature_names[3])
_ = plt.legend()

Note that the synthetic points are **similar** to the original ones and lie close to all the classes. However, this is not the only approach to creating synthetic data for the prediction. New data can also be generated **only in the neighbourhood** of the explained point. In the official LIME implementation the hyperparameter `sample_around_instance` is responsible for this.


We, on the other hand, will implement the approach where the generation of synthetic points is done on the basis of **all** the data.

Next let us move on to the key things we are studying: the process of discretisation and binarisation of the data. \

**Discretisation** is the process of transforming some continuous function into a discrete one. In this case (and this is one of the possible out-of-the-box implementations of the LIME algorithm), discretisation happens by splitting into *quartiles*.


**Quartiles** are the values that split an ordered dataset into four equal parts, each of which contains 25% of the data. The process of discretisation (splitting) into quartiles is called quartile discretisation.
It happens as follows:
1. We extract the boundaries of each quartile
2. We encode every point in the dataset as a vector with coordinates from the set {0, 1, 2, 3} by the rule — we replace the value of the coordinate with the number of the quartile (0, 1, 2, 3) it falls into.

**Quiz 2:** What coordinates will the vector $[2.3, 5.1, 2., 3.3]$ get if the quartile boundaries are the following: $[0, 2.1], [2.2, 3], [3.1, 4], [4.1, 5.3]$

As the answer write down coordinate number 2 (the coordinates are numbered from one).

In [ ]:
discretiser = fatf_discretisation.QuartileDiscretiser(
    X_train,
    feature_names=iris_feature_names)

data_point_discretised = discretiser.discretise(data_point)
sampled_data_discretised = discretiser.discretise(sampled_data)

discretiser.feature_bin_boundaries

Let us make sure that the boundary values really are quartiles.

In [ ]:
import pandas as pd

pd.DataFrame(X_train, columns=iris_feature_names).describe()[3:] #Indeed, the boundaries coincide with the statistics we need.

In [ ]:
#Let us look at the discretised point
data_point_discretised

Now, in order to obtain an *interpretable model*, let us carry out the **binarisation of the data**.
Binarisation of the data — as the name suggests — is bringing all the coordinates to a combination of zeros and ones (a binary form). We will binarise by the following rule: put 0 if the coordinates of the discretised data point do not coincide with the data point from the sample, and 1 otherwise.

In [ ]:
import fatf.utils.data.transformation as fatf_transformation

sampled_data_binarised = fatf_transformation.dataset_row_masking(
    sampled_data_discretised, data_point_discretised)

fatf_transformation.dataset_row_masking(data_point_discretised.reshape(1, -1), data_point_discretised)

In practice binarisation is carried out for text data and images. On our example it will give results that are inconsistent with other approaches to interpretation. Which ones are better has to be checked in practice.

Let us look at all 3 transformations once again:

In [ ]:
print(f'Original data point: {data_point}')
print(f'Discretised data point: {data_point_discretised}')

print(f'Some instance from the discretised points before binarisation: {sampled_data_discretised[7]}')
print(f'Some instance from the discretised points after binarisation: {sampled_data_binarised[7]}')

As the last step it remains to compute the weights of the objects for the loss function. We will simply do this by the corresponding formula, which you recalled in the task above.

In [ ]:
import fatf.utils.distances as fatf_distances
import fatf.utils.kernels as fatf_kernels

features_number = sampled_data_binarised.shape[1]
kernel_width = np.sqrt(features_number) * 0.75

distances = fatf_distances.euclidean_point_distance(np.ones(features_number), sampled_data_binarised) #we will compute the distances on the basis of the binarised data
weights = fatf_kernels.exponential_kernel(
     distances, width=kernel_width)

## **Training the local algorithm**

The task solved by the models for the Fisher irises dataset is a multiclass classification task. Thus, when training a surrogate model we can train it in two ways:
- using the ONE VS REST approach, where the surrogate model will predict 0 or 1 depending on whether the object belongs to the class we are interested in.
- using the classical approach, where the surrogate model will predict a vector of probabilities.

The advantage of the first approach is the focus on the class we are interested in, that of the second one is universality — the resulting surrogate can be used to explain an object from any class.

Besides, above we noted that the model can be trained on binarised data or not. Training on binarised data gives a model that answers the question:

*"If this particular feature value of the explained data point were outside the range (for numerical features) or had a different value (for a categorical feature), how would that affect the probability of this point belonging to the explained class (probabilistic classification) / the predicted numerical value (regression)?"*

This is useful for images, since it allows us to compare the importance of different segments of the picture.

In our case, though, the model will solve the task of minimising the deviation of the predicted values from the true ones in the classical sense.

In [ ]:
sampled_data_predictions_versicolor = sampled_data_probabilities[:, 1] # let us keep the probabilities for the OVR approach

Let us initialise the model and train it on the discretised data on all the probabilities.

In [ ]:
import sklearn.linear_model
import numpy as np

In [ ]:
lime_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)

lime_model.fit(sampled_data_discretised, sampled_data_predictions, sample_weight=weights)
for name, importance in zip(iris_feature_names, lime_model.coef_):
     print('->{}<-: {}'.format(name, importance))

**And there you go, you have trained your first LIME!**
Let us fix the important features in order: `petal width`, `petal lenght`

Let us look at the variant implemented on non-discretised data

In [ ]:
lime_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)

lime_model.fit(sampled_data, sampled_data_predictions, sample_weight=weights)  #let us look at the variant implemented on non-discretised data

for name, importance in zip(iris_feature_names, lime_model.coef_):
     print('->{}<-: {}'.format(name, importance))

Let us fix the important features in order here: petal width, sepal lenght

**Quiz 3: Change the approach to computing the distances — compute them on the basis of the original data values. Train the local model on the discretised data. Have the 2 most important features changed?**

In [ ]:
# Your code here

**The library implementation**. \
Let us look at what the library implementation gives. All the hyperparameters are fairly intuitive, but just in case let us explain each of them:

- `class_names` — the names of the predicted classes
- `feature_names` — the names of the features
- `kernel_width`— the kernel width, by default (and in our case) $\sqrt{n\_features}*0.75$
- `verbose` — detailed output while training the surrogate model
- `discretizer` — the approach to discretisation
- `mode` — the task solved by the model
- `discretize_continuous` — whether the features have to be discretised
- `sample_around_instance` — whether to generate synthetic data **only in the neighbourhood** of the point under consideration

Let us look at the important features on discretised and non-discretised data one after another. We start with the non-discretised ones.  

In [ ]:
from lime.lime_tabular import LimeTabularExplainer #let us look at what the library implementation gives

explainer = LimeTabularExplainer(X_train,
                                 class_names=iris_target_names,
                                 feature_names=iris_feature_names,
                                 kernel_width=np.sqrt(features_number) * 0.75,
                                 verbose=False,
                                 mode='classification',
                                 discretize_continuous=False,
                                 sample_around_instance=False)

exp = explainer.explain_instance(data_point, blackbox_model.predict_proba, top_labels=1)
exp.show_in_notebook()

We see that without discretisation there is no clear strength of influence of particular features visible.

**Quiz 4: Obtain the interpretation with quartile discretisation. Is it consistent with the one obtained above with the manual implementation?**

In [ ]:
# Your code here

We see that the results are not exactly equal to each other, but they are similar in their general conclusions.

**Quiz 5: Obtain the interpretation with the help of the manual implementation on the binarised data with the weights `weights`. Which feature (features) are highlighted in this case?**

In [ ]:
# Your code here

## **A surrogate tree**

A linear model is not the only one in our arsenal. In some cases building a surrogate tree turns out to be more useful and informative. A simple example of the construction:

In [ ]:
import sklearn.tree

blimey_tree = sklearn.tree.DecisionTreeClassifier(max_depth=3, random_state=42)
blimey_tree.fit(sampled_data, sampled_data_predictions, sample_weight=weights)


In [ ]:
for n_i in zip(iris_feature_names, blimey_tree.feature_importances_):
     name, importance = n_i
     print('->{}<-: {}'.format(name, importance))


Also, for the interpretation it is useful here to visualise the splitting rules.

In [ ]:
from sklearn import tree
print(tree.export_text(blimey_tree))

Or in a nicer way.

In [ ]:
import graphviz

dot_data = tree.export_graphviz(blimey_tree, out_file=None,
                                feature_names=iris_feature_names,
                                class_names=iris_target_names,
                                filled=True)

# Draw graph
graph = graphviz.Source(dot_data, format="png")
graph

**Quiz 6: Which feature is missing from the structure of the surrogate tree we built?**

### **Conclusions**
- LIME allows different approaches to forming and estimating feature importance
- The explanations inside different LIME scenarios can differ, so the truth has to be checked empirically, and the hypotheses have to be generated on the basis of a combination of methods/models
- Using LIME assumes that the features used are conceptually understandable in the first place (pieces of an image, values of a numerical vector that correspond to real data)